In [55]:
import os
import pandas as pd

# Set pandas display options to show all rows and columns without truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

def create_t50_dataframe(directory_path):
    """
    Reads all .txt files from a directory, parses them into full context and quote,
    and returns them as a pandas DataFrame.
    
    The expected file format is:
    [Full Context Text]
    @highlight
    [Quote Text]
    """
    data = []
    
    # Ensure the directory exists
    if not os.path.isdir(directory_path):
        print(f"Directory not found: {directory_path}")
        return None
    
    # Iterate through all files in the directory, sorting them to ensure a consistent order
    for filename in sorted(os.listdir(directory_path)):
        if filename.startswith("index_") and filename.endswith(".txt"):
            file_path = os.path.join(directory_path, filename)
            
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                # Split the content at the '@highlight' marker
                parts = content.split('@highlight')
                
                if len(parts) == 2:
                    full_context = parts[0].strip()
                    quote = parts[1].strip()
                    data.append({'T50 full context': full_context, 'T50 quote': quote})
                else:
                    print(f"Warning: Could not parse file {filename} as expected. It does not contain '@highlight'.")
            except Exception as e:
                print(f"Error reading or processing file {filename}: {e}")
                
    # Create a DataFrame from the collected data
    df = pd.DataFrame(data)
    return df

# --- Main Execution ---

# Define the path to the T50 directory
t50_directory = '../data/interim/t50/T50'

# Create the DataFrame
t50_df = create_t50_dataframe(t50_directory)

# Display the first 5 rows and the shape of the DataFrame to verify
if t50_df is not None:
    print("DataFrame created successfully.")
    display(t50_df.head(2))
    print(f"\nDataFrame shape: {t50_df.shape}")

DataFrame created successfully.


,T50 full context,T50 quote
0,"-- A dentist 's question . 30 Errors of haste are seldom committed singly . The first time a man always docs too much . And precisely on that account he commits a second error , and then he does too little . 31 The trodden worm curls up . This testifies to its caution . It thus reduces its chances of being trodden upon again . In the language of morality : Humility . -- 32 There is such a thing as a hatred of lies and dissimulation , which is the outcome of a delicate sense of humour ; there is also the selfsame hatred but as the result of cowardice , in so far as falsehood is forbidden by Divine law . Too cowardly to lie .... 33 What trifles constitute happiness ! The sound of a bagpipe . Without music life would be a mistake . The German imagines even God as a songster . 34 _ On ne peut penser et écrire qu'assis _ ( G. Flaubert ) . Here I have got you , you nihilist ! A sedentary life is the real sin against the Holy Spirit . Only those thoughts that come by walking have any value . 35 There are times when we psychologists are like horses , and grow fretful . We see our own shadow rise and fall before us . The psychologist must look away from himself if he wishes to see anything at all . 36 Do we immoralists injure virtue in any way ? Just as little as the anarchists injure royalty .","Without music , life would be a mistake ."
1,"She and your brother choose to go , and you will be only getting ill will . ” Catherine submitted , and though sorry to think that Isabella should be doing wrong , felt greatly relieved by Mr. Allen ’s approbation of her own conduct , and truly rejoiced to be preserved by his advice from the danger of falling into such an error herself . Her escape from being one of the party to Clifton was now an escape indeed ; for what would the Tilneys have thought of her , if she had broken her promise to them in order to do what was wrong in itself , if she had been guilty of one breach of propriety , only to enable her to be guilty of another ? CHAPTER 14 The next morning was fair , and Catherine almost expected another attack from the assembled party . With Mr. Allen to support her , she felt no dread of the event : but she would gladly be spared a contest , where victory itself was painful , and was heartily rejoiced therefore at neither seeing nor hearing anything of them . The Tilneys called for her at the appointed time ; and no new difficulty arising , no sudden recollection , no unexpected summons , no impertinent intrusion to disconcert their measures , my heroine was most unnaturally able to fulfil her engagement , though it was made with the hero himself . They determined on walking round Beechen Cliff , that noble hill whose beautiful verdure and hanging coppice render it so striking an object from almost every opening in Bath . “ I never look at it , ” said Catherine , as they walked along the side of the river , “ without thinking of the south of France . ” “ You have been abroad then ? ” said Henry , a little surprised . “ Oh ! No , I only mean what I have read about . It always puts me in mind of the country that Emily and her father travelled through , in The Mysteries of Udolpho . But you never read novels , I dare say ? ” “ Why not ? ” “ Because they are not clever enough for you — gentlemen read better books . ” “ The person , be it gentleman or lady , who has not pleasure in a good novel , must be intolerably stupid . I have read all Mrs. Radcliffe ’s works , and most of them with great pleasure . The Mysteries of Udolpho , when I had once begun it , I could not lay down again ; I remember finishing it in two days — my hair standing on end the whole time . ” “ Yes , ” added Miss Tilney , “ and I remember that you undertook to read it aloud to me , and that when I was called away for only five minutes to answer a note , instead of waiting for me , you took the volume into the Hermitage Walk , and I was obliged to stay till you had finished it . ” “ Thank you 


DataFrame shape: (5015, 2)


In [56]:
import os
import pandas as pd
import nltk
from thefuzz import fuzz
import re
import ftfy

# --- Configuration ---
# Set pandas display options to show all content without truncation
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 300)

# The thresholds we determined to be optimal
GLOBAL_MATCH_CUTOFF = 65
PRECISE_SIMILARITY_CUTOFF = 70

# --- Step 1: Load the Initial Data ---

def create_t50_dataframe(directory_path):
    """
    Reads all .txt files from a directory, parses them into full context and quote,
    and returns them as a pandas DataFrame.
    """
    data = []
    if not os.path.isdir(directory_path):
        print(f"Directory not found: {directory_path}")
        return None
    
    print(f"Reading data from {directory_path}...")
    for filename in sorted(os.listdir(directory_path)):
        if filename.startswith("index_") and filename.endswith(".txt"):
            file_path = os.path.join(directory_path, filename)
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                parts = content.split('@highlight')
                if len(parts) == 2:
                    full_context, quote = parts[0].strip(), parts[1].strip()
                    data.append({'T50 full context': full_context, 'T50 quote': quote})
                else:
                    print(f"Warning: Could not parse file {filename}.")
            except Exception as e:
                print(f"Error reading or processing file {filename}: {e}")
                
    return pd.DataFrame(data)

# Define the absolute path to the T50 directory
t50_directory = '../data/interim/t50/T50'

# Create the initial DataFrame
t50_df = create_t50_dataframe(t50_directory)
print(f"Successfully loaded {len(t50_df)} rows.")

# --- Step 2: Calculate Fuzzy Match Scores ---

print("\nCalculating fuzzy match scores for all rows...")
t50_df['fuzzy_match_score'] = t50_df.apply(
    lambda row: fuzz.partial_ratio(str(row['T50 quote']), str(row['T50 full context'])),
    axis=1
)
print("Fuzzy match scoring complete.")

# --- Step 3: Define the Final, Precise Cleaning Logic ---

# Ensure NLTK's sentence tokenizer is available
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    print("Downloading NLTK's 'punkt' sentence tokenizer...")
    nltk.download('punkt')

def normalize_text(text):
    """Fixes unicode errors, lowercases, and removes punctuation for better matching."""
    if not isinstance(text, str): return ""
    text = ftfy.fix_text(text)
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def apply_final_cleaning(row):
    """
    Contains the final, precise logic. It judges each sentence independently
    and removes only those with a high similarity to the quote.
    """
    if row['fuzzy_match_score'] < GLOBAL_MATCH_CUTOFF:
        return row['T50 full context']

    quote_norm = normalize_text(row['T50 quote'])
    context_orig = str(row['T50 full context'])
    
    sentences_orig = nltk.sent_tokenize(context_orig)
    if not sentences_orig:
        return ""
        
    sentences_norm = [normalize_text(s) for s in sentences_orig]
    sentence_scores = [fuzz.token_set_ratio(quote_norm, s) for s in sentences_norm]

    # Keep only the sentences that DO NOT meet the high similarity threshold.
    kept_sentences = [
        sentences_orig[i] for i, score in enumerate(sentence_scores) 
        if score < PRECISE_SIMILARITY_CUTOFF
    ]
    
    return " ".join(kept_sentences)

# --- Step 4: Apply Cleaning and Rename the Column ---

print("\nApplying final cleaning logic to create the new column...")
t50_df['T50 quote-free context'] = t50_df.apply(apply_final_cleaning, axis=1)
rows_changed = (t50_df['T50 full context'] != t50_df['T50 quote-free context']).sum()

print("\n✅ Final DataFrame is ready.")
print(f"{rows_changed} rows had their context cleaned.")

# --- Final Verification ---
print("\nDisplaying the first 10 rows of the final result:")
display(t50_df[['T50 quote', 'T50 full context', 'T50 quote-free context', 'fuzzy_match_score']].head(10))

Reading data from ../data/interim/t50/T50...
Successfully loaded 5015 rows.

Calculating fuzzy match scores for all rows...
Fuzzy match scoring complete.

Applying final cleaning logic to create the new column...

✅ Final DataFrame is ready.
4953 rows had their context cleaned.

Displaying the first 10 rows of the final result:


,T50 quote,T50 full context,T50 quote-free context,fuzzy_match_score
0,"Without music , life would be a mistake .","-- A dentist 's question . 30 Errors of haste are seldom committed singly . The first time a man always docs too much . And precisely on that account he commits a second error , and then he does too little . 31 The trodden worm curls up . This testifies to its caution . It thus reduces its chanc...","-- A dentist 's question . 30 Errors of haste are seldom committed singly . The first time a man always docs too much . And precisely on that account he commits a second error , and then he does too little . 31 The trodden worm curls up . This testifies to its caution . It thus reduces its chanc...",95
1,"The person , be it gentleman or lady , who has not pleasure in a good novel , must be intolerably stupid .","She and your brother choose to go , and you will be only getting ill will . ” Catherine submitted , and though sorry to think that Isabella should be doing wrong , felt greatly relieved by Mr. Allen ’s approbation of her own conduct , and truly rejoiced to be preserved by his advice from the dan...","She and your brother choose to go , and you will be only getting ill will . ” Catherine submitted , and though sorry to think that Isabella should be doing wrong , felt greatly relieved by Mr. Allen ’s approbation of her own conduct , and truly rejoiced to be preserved by his advice from the dan...",100
2,Doubt thou the stars are fire ;\nDoubt that the sun doth move ;\nDoubt truth to be a liar ;\nBut never doubt I love .,"Mad let us grant him then . And now remains That we find out the cause of this effect , Or rather say , the cause of this defect , For this effect defective comes by cause . Thus it remains , and the remainder thus . Perpend , I have a daughter — have whilst she is mine — Who in her duty and obe...","Mad let us grant him then . And now remains That we find out the cause of this effect , Or rather say , the cause of this defect , For this effect defective comes by cause . Thus it remains , and the remainder thus . Perpend , I have a daughter — have whilst she is mine — Who in her duty and obe...",95
3,"I am the happiest creature in the world . Perhaps other people have said so before , but not one with such justice . I am happier even than Jane ; she only smiles , I laugh .","My avowed one , or what I avowed to myself , was to see whether your sister were still partial to Bingley , and if she were , to make the confession to him which I have since made . ” “ Shall you ever have courage to announce to Lady Catherine what is to befall her ? ” “ I am more likely to want...","My avowed one , or what I avowed to myself , was to see whether your sister were still partial to Bingley , and if she were , to make the confession to him which I have since made . ” “ Shall you ever have courage to announce to Lady Catherine what is to befall her ? ” “ I am more likely to want...",100
4,"Each one of us is alone in the world . He is shut in a tower of brass , and can communicate with his fellows only by signs , and the signs have no common value , so that their sense is vague and uncertain . We seek pitifully to convey to others the treasures of our heart , but they have not the ...","I saw a tormented spirit striving for the release of expression . I turned to him . "" I wonder if you have n't mistaken your medium , "" I said . "" What the hell do you mean ? "" "" I think you 're trying to say something , I do n't quite know what it is , but I 'm not sure that the best way of say...","I saw a tormented spirit striving for the release of expression . I turned to him . "" I wonder if you have n't mistaken your medium , "" I said . "" What the hell do you mean ? "" "" I think you 're trying to say something , I do n't quite know what it is , but I 'm not sure that the best way of say...",100
5,"There 's no beauty without poignancy and there 's no poignancy without the feeling that it 's going , men , names , book

In [57]:
import spacy
from tqdm import tqdm
from spacy.tokens import Doc

# --- Configuration for Sampling Logic ---
# These parameters control the new, smarter sampling logic.
# They are placed here for easy access and modification.
LOW_THRESHOLD = 0.7   # The snippet can be as short as 70% of the quote's length.
HIGH_THRESHOLD = 1.3  # The snippet can be as long as 130% of the quote's length.
MAX_HIGH_GUARDRAIL = 1.5 # The absolute maximum length allowed (150%) before discarding.


def best_snippet(doc: Doc, target_L: int):
    """
    Return a sentence-aligned snippet from a spaCy Doc object whose word count
    is as close as possible to target_L, subject to bounds.
    This is a deterministic function for reproducibility.
    """
    sents = [sent.text for sent in doc.sents]
    lengths = [len(s.split()) for s in sents]
    
    if not sents:
        return None

    best = None
    best_diff = float("inf")
    lower_bound = int(LOW_THRESHOLD * target_L)
    upper_bound = int(HIGH_THRESHOLD * target_L)
    absolute_max = int(MAX_HIGH_GUARDRAIL * target_L) + 1 # Add 1 for safety with rounding
    
    # --- Primary Search: Find best contiguous block within thresholds ---
    for i in range(len(sents)):
        current_len = 0
        for j in range(i, len(sents)):
            current_len += lengths[j]
            
            # Check if the current block is a candidate
            if lower_bound <= current_len <= upper_bound:
                diff = abs(current_len - target_L)
                if diff < best_diff:
                    best = " ".join(sents[i:j+1])
                    best_diff = diff
            
            # Optimization: If we are already way over the absolute max, stop building this block
            if current_len > absolute_max + 50:
                 break
        
        # Early exit if we find a perfect match
        if best_diff == 0:
            break
    
    # --- Fallback: If no block was found, find the single best sentence ---
    if best is None:
        # Check if there are any sentences to process
        if not lengths:
            return None
        diffs = [abs(l - target_L) for l in lengths]
        idx = int(min(range(len(diffs)), key=diffs.__getitem__))
        best = sents[idx]
    
    # --- Final Guard Rail: Discard if the final result is too long OR too short ---
    final_len = len(best.split())
    # The snippet must not be excessively long
    if final_len > absolute_max:
        return None
    # After all fallbacks, it must still be reasonably close (at least half the quote's length)
    # This catches the cases where a very long quote gets a tiny fallback snippet.
    if final_len < target_L * 0.5:
         return None
        
    return best

# Load spacy model
print("Loading spaCy model 'en_core_web_sm'...")
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spacy model 'en_core_web_sm'...")
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded successfully.")

# --- Optimized Processing with nlp.pipe ---
print("Processing 'T50 quote-free context' with spaCy using nlp.pipe for efficiency...")

# Get texts, ensuring they are strings to prevent errors with spaCy
context_texts = t50_df["T50 quote-free context"].fillna('').astype(str).tolist()

# Process texts in batches using multiple cores for a significant speedup
docs = list(tqdm(nlp.pipe(context_texts, n_process=-1), total=len(context_texts), desc="SpaCy Processing"))

print("\nGenerating matched snippets for 'T50 quote-free context'...")
snippets = []
# Loop through the dataframe and the pre-processed docs together
for i, row in tqdm(t50_df.iterrows(), total=len(t50_df), desc="Generating Snippets"):
    doc = docs[i]
    quote_text = row.get("T50 quote")
    # Also check the original page text column for missing values
    context_text = row.get("T50 quote-free context")

    if pd.isna(quote_text) or pd.isna(context_text) or not context_text:
        snippets.append(None)
        continue
        
    quote_len = len(str(quote_text).split())
    
    snippet = best_snippet(doc, quote_len)
    snippets.append(snippet)
    
t50_df['T50 Quote-Free Context Length Matched'] = snippets

# --- Final Analysis ---
# Drop rows where a snippet couldn't be generated (outliers)
rows_before = len(t50_df)
t50_df.dropna(subset=['T50 Quote-Free Context Length Matched'], inplace=True)
rows_after = len(t50_df)

discarded_count = rows_before - rows_after

print("\n--- Processing Complete ---")
print(f"Successfully generated snippets for {rows_after} out of {rows_before} rows.")
if discarded_count > 0:
    print(f"Discarded {discarded_count} rows that did not meet the length criteria.")

display(t50_df[['T50 quote', 'T50 quote-free context', 'T50 Quote-Free Context Length Matched']].head())


Loading spaCy model 'en_core_web_sm'...
spaCy model loaded successfully.
Processing 'T50 quote-free context' with spaCy using nlp.pipe for efficiency...


SpaCy Processing:   0%|          | 0/5015 [00:00<?, ?it/s]python(8855) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8857) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8859) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8865) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8866) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8881) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8897) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(8901) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
SpaCy Processing: 100%|██████████| 5015/5015 [02:17<00:00, 36.60it/s] 



Generating matched snippets for 'T50 quote-free context'...


Generating Snippets: 100%|██████████| 5015/5015 [00:02<00:00, 2087.24it/s]



--- Processing Complete ---
Successfully generated snippets for 5000 out of 5015 rows.
Discarded 15 rows that did not meet the length criteria.


,T50 quote,T50 quote-free context,T50 Quote-Free Context Length Matched
0,"Without music , life would be a mistake .","-- A dentist 's question . 30 Errors of haste are seldom committed singly . The first time a man always docs too much . And precisely on that account he commits a second error , and then he does too little . 31 The trodden worm curls up . This testifies to its caution . It thus reduces its chanc...",30 Errors of haste are seldom committed singly .
1,"The person , be it gentleman or lady , who has not pleasure in a good novel , must be intolerably stupid .","She and your brother choose to go , and you will be only getting ill will . ” Catherine submitted , and though sorry to think that Isabella should be doing wrong , felt greatly relieved by Mr. Allen ’s approbation of her own conduct , and truly rejoiced to be preserved by his advice from the dan...","It always puts me in mind of the country that Emily and her father travelled through , in The Mysteries of Udolpho ."
2,Doubt thou the stars are fire ;\nDoubt that the sun doth move ;\nDoubt truth to be a liar ;\nBut never doubt I love .,"Mad let us grant him then . And now remains That we find out the cause of this effect , Or rather say , the cause of this defect , For this effect defective comes by cause . Thus it remains , and the remainder thus . Perpend , I have a daughter — have whilst she is mine — Who in her duty and obe...","Perpend , I have a daughter — have whilst she is mine — Who in her duty and obedience , mark , Hath given me this ."
3,"I am the happiest creature in the world . Perhaps other people have said so before , but not one with such justice . I am happier even than Jane ; she only smiles , I laugh .","My avowed one , or what I avowed to myself , was to see whether your sister were still partial to Bingley , and if she were , to make the confession to him which I have since made . ” “ Shall you ever have courage to announce to Lady Catherine what is to befall her ? ” “ I am more likely to want...","“ I am more likely to want time than courage , Elizabeth . But it ought to be done , and if you will give me a sheet of paper , it shall be done directly . ”"
4,"Each one of us is alone in the world . He is shut in a tower of brass , and can communicate with his fellows only by signs , and the signs have no common value , so that their sense is vague and uncertain . We seek pitifully to convey to others the treasures of our heart , but they have not the ...","I saw a tormented spirit striving for the release of expression . I turned to him . "" I wonder if you have n't mistaken your medium , "" I said . "" What the hell do you mean ? "" "" I think you 're trying to say something , I do n't quite know what it is , but I 'm not sure that the best way of say...","I turned to him . "" I wonder if you have n't mistaken your medium , "" I said . "" What the hell do you mean ? "" "" I think you 're trying to say something , I do n't quite know what it is , but I 'm not sure that the best way of saying it is by means of painting . "" When I imagined that on seeing ..."


In [58]:
import pandas as pd

# Define the paths for the popular quotes and the quotes to be used for deduplication
popular_quotes_file = '../data/interim/cleaning_data/Data/goodreads-english-popular-quotes.csv'
quotes_to_exclude_file = '../data/interim/google_books_work/quotes_pages_snippets_cleaned.csv'

try:
    # Load both the popular quotes and the exclusion list
    popular_quotes_df = pd.read_csv(popular_quotes_file)
    merged_quotes_df = pd.read_csv(quotes_to_exclude_file)
    
    # Create a set of the quotes from quotes_pages_snippets_cleaned.csv for efficient lookup
    # A set provides much faster lookups than a list or a DataFrame column.
    quotes_to_exclude = set(merged_quotes_df['QUOTE'])
    
    # Sort the popular quotes by likes in descending order
    sorted_popular_df = popular_quotes_df.sort_values(by='LIKES', ascending=False)
    
    # Remove rows from sorted_popular_df where the 'QUOTE' is in our exclusion set
    # The .isin() method checks for each quote's presence in the set.
    # The '~' operator inverts this, keeping only the quotes that are NOT in the set.
    deduplicated_popular_df = sorted_popular_df[~sorted_popular_df['QUOTE'].isin(quotes_to_exclude)]
    
    # Now, take the top 6000 from this newly cleaned and sorted DataFrame
    top_6k_deduplicated_df = deduplicated_popular_df.head(6000)[['QUOTE', 'LIKES']].copy()
    
    # Rename the columns for clarity
    top_6k_deduplicated_df.rename(columns={'QUOTE': 'Goodreads Popular Quote', 'LIKES': 'Likes of Popular Quote'}, inplace=True)
    
    # Display the head and shape of the final DataFrame
    print("Deduplication successful.")
    print(f"Original number of popular quotes: {len(sorted_popular_df)}")
    print(f"Number of popular quotes after deduplication: {len(deduplicated_popular_df)}")
    
    display(top_6k_deduplicated_df.head())
    print(f"\nFinal DataFrame shape: {top_6k_deduplicated_df.shape}")

except FileNotFoundError as e:
    if str(e) == popular_quotes_file:
        print(f"Error: The popular quotes file was not found at: {popular_quotes_file}")
    elif str(e) == quotes_to_exclude_file:
        print(f"Error: The quotes to exclude file was not found at: {quotes_to_exclude_file}")
    else:
        print(f"An error occurred: {e}")
except KeyError as e:
    print(f"Error: A required column was not found in one of the files. Missing column: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")


Deduplication successful.
Original number of popular quotes: 225262
Number of popular quotes after deduplication: 223758


,Goodreads Popular Quote,Likes of Popular Quote
0,"If you want to know what a man's like, take a good look at how he treats his inferiors, not his equals.",95186
1,"Friendship ... is born at the moment when one man says to another ""What! You too? I thought that no one but myself . . .",83698
2,I am so clever that sometimes I don't understand a single word of what I am saying.,67183
3,"Without music, life would be a mistake.",66263
4,We accept the love we think we deserve.,64699



Final DataFrame shape: (6000, 2)


In [59]:
import pandas as pd
import numpy as np

# Define the file path and the columns to drop
file_path = '../data/interim/google_books_work/quotes_pages_snippets_cleaned.csv'
columns_to_drop = ['date_fetch_status', 'preview_link', 'source_method', 'Filename', 'TAGS']

try:
    # --- 1. Load the Data ---
    print(f"Loading data from {file_path}...")
    full_dataset_df = pd.read_csv(file_path)
    print("Data loaded successfully.")

    # --- 2. Clean and Prepare the Data ---
    
    # Drop the specified columns
    full_dataset_df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
    
    # Rename the 'MATCHED_SNIPPET' column
    full_dataset_df.rename(columns={'MATCHED_SNIPPET': 'Google Books Length Matched Snippet'}, inplace=True)
    
    # --- 3. Implement Priority Sampling ---
    
    print("Performing priority sampling...")
    # Create a copy to avoid SettingWithCopyWarning
    df_to_sample = full_dataset_df.copy()

    # Identify rows with complete metadata
    invalid_metadata_values = ['Unknown', 'Not Found']
    priority_rows = df_to_sample[
        (~df_to_sample['language_or_country'].isin(invalid_metadata_values)) &
        (~df_to_sample['publication_date'].isin(invalid_metadata_values))
    ]

    # Rows with incomplete metadata
    other_rows = df_to_sample[
        (df_to_sample['language_or_country'].isin(invalid_metadata_values)) |
        (df_to_sample['publication_date'].isin(invalid_metadata_values))
    ]
    
    print(f"Found {len(priority_rows)} rows with complete metadata and {len(other_rows)} rows with incomplete metadata.")

    # --- 4. Sample 5,300 Rows ---
    
    target_sample_size = 5300
    
    if len(priority_rows) >= target_sample_size:
        # If we have enough priority rows, sample directly from them
        final_sample_df = priority_rows.sample(n=target_sample_size, random_state=42)
        print(f"Sampled {target_sample_size} rows directly from the priority group.")
    else:
        # If not, take all priority rows and sample the remainder
        print("Not enough priority rows. Taking all priority rows first.")
        remaining_needed = target_sample_size - len(priority_rows)
        
        if len(other_rows) >= remaining_needed:
            other_sample = other_rows.sample(n=remaining_needed, random_state=42)
            final_sample_df = pd.concat([priority_rows, other_sample])
            print(f"Sampled the remaining {remaining_needed} rows from the incomplete metadata group.")
        else:
            final_sample_df = pd.concat([priority_rows, other_rows])
            print(f"Warning: Total available rows ({len(final_sample_df)}) is less than target of {target_sample_size}.")

    # Shuffle the final combined DataFrame
    final_sample_df = final_sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

    # --- 5. Filter by Length (New Section) ---
    print("\nFiltering sample by quote length...")
    
    # Ensure the text columns are strings to prevent errors with .str accessor
    final_sample_df['QUOTE'] = final_sample_df['QUOTE'].astype(str)
    final_sample_df['Google Books Length Matched Snippet'] = final_sample_df['Google Books Length Matched Snippet'].astype(str)
    
    # Calculate lengths of the text columns
    final_sample_df['quote_length'] = final_sample_df['QUOTE'].str.split().str.len()
    final_sample_df['matched_snippet_length'] = final_sample_df['Google Books Length Matched Snippet'].str.split().str.len()

    # Define the minimum word count and apply the filter
    min_word_count = 3
    pre_filter_count = len(final_sample_df)
    final_sample_df = final_sample_df[final_sample_df['quote_length'] >= min_word_count].copy() # Use .copy()
    
    print(f"Removed {pre_filter_count - len(final_sample_df)} rows with quotes shorter than {min_word_count} words.")

    # --- 6. Display Final Results ---
    print("\nGoogle Books Snippet sampling and filtering complete.")
    display(final_sample_df.head(2))
    print(f"\nFinal DataFrame shape after length filtering: {final_sample_df.shape}")

except FileNotFoundError:
    print(f"Error: The file was not found at the specified path: {file_path}")
except KeyError as e:
    print(f"Error: A required column was not found in the CSV file. Missing column: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Loading data from ../data/interim/google_books_work/quotes_pages_snippets_cleaned.csv...
Data loaded successfully.
Performing priority sampling...
Found 4935 rows with complete metadata and 323 rows with incomplete metadata.
Not enough priority rows. Taking all priority rows first.

Filtering sample by quote length...
Removed 13 rows with quotes shorter than 3 words.

Google Books Snippet sampling and filtering complete.


,LIKES,AUTHOR,TITLE,language_or_country,publication_date,genres,QUOTE,Google Books Page Text,Google Books Length Matched Snippet,quote_length,matched_snippet_length
0,1,Patrick Mcgrath,Dr. Haggard'S Disease,United Kingdom,1993,Fiction,"Houses, I have come to believe, like love, like nature herself, should not reassure, should not attempt to soothe, or give comfort, but should, rather, .","It was at a funeral I first saw her, did she ever tell you that? And do\nyou know, I can't remember whose it was! Who was dead, I mean. It\nwas October 1937, a fine, crisp day, and the air in London had a sort\nof smoky quality to it. The leaves drifting off the chestnuts along\nJubilee Road hea...","Who was dead, I mean. It\nwas October 1937, a fine, crisp day, and the air in London had a sort\nof smoky quality to it.",26,26
1,1,Richard Kadrey,The Kill Society,United States,2017,Fiction,"\r\n It's not something you think about, it just happens. You fall into the orbit of friends and familiar faces. You don't even have to like each other. You just have to be there to remind each other that you survived and that this is real. I'm sure there's a scientific name for it....","\r\n SO FAR, BEING dead is about as much fun as a barbed-wire G-string.\r\nYes, there is such a thing. They invented it in Hell, which is\r\nwhere I am. I already said I was dead. Where else would I be? Try\r\nto keep up.\r\nWhere was I? I was talking about fun. First off, there's t...","I kick a rock down the slope and follow it as it tumbles ahead\r\nof me into the valley. As it goes, I spot something on the trail\r\nahead. Bend down to pick it up. Okay, I might not know where I\r\nam, but I know I'm being fucked with. What I'm holding is a dusty\r\npack of Maledictions. But n...",62,62



Final DataFrame shape after length filtering: (5245, 11)


In [60]:
%pip install datasets

python(8955) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [61]:
import pandas as pd
import nltk
from datasets import load_dataset
import random
import re

# --- Setup and Configuration ---
random.seed(42)
# The tolerance allows for wiggle room in matching lengths. 
# 0.1 means the found snippet can be +/- 10% of the target length.
LENGTH_TOLERANCE = 0.1 

try:
    nltk.data.find('corpora/brown')
    print("NLTK data (Brown corpus) already available.")
except nltk.downloader.DownloadError:
    print("Downloading NLTK data (Brown corpus)...")
    nltk.download('brown')
    print("Download complete.")

# --- NEW: More Flexible Helper Function ---

def find_flexible_contiguous_matches(source_words, target_lengths, source_name, tolerance=0.1):
    """
    Finds contiguous blocks of text that are within a tolerance of the target lengths
    and adhere to sentence boundary rules.
    """
    matches = []
    current_pos = 0
    total_words = len(source_words)

    for i, target_len in enumerate(target_lengths):
        if target_len == 0: continue # Skip zero-length targets

        # Define the acceptable length range
        lower_len = int(target_len * (1 - tolerance))
        upper_len = int(target_len * (1 + tolerance))
        
        found_match = False
        # We'll search for a valid STARTING word first
        for start_attempt in range(500): # Max 500 attempts to find a starting word
            start_pos = current_pos + start_attempt
            
            if start_pos >= total_words: break # End of source text
            
            # RULE 1: Must start with a capital letter
            if not source_words[start_pos][0].isupper():
                continue

            # If we found a good start, now search for a good END within the tolerance window
            for end_offset in range(lower_len, upper_len + 1):
                end_pos = start_pos + end_offset
                if end_pos >= total_words: break

                # RULE 2: Must end with sentence-ending punctuation
                if source_words[end_pos - 1].endswith(('.', '?', '!')):
                    chunk_words = source_words[start_pos:end_pos]
                    matches.append(" ".join(chunk_words))
                    current_pos = end_pos # IMPORTANT: Prevents re-using sections
                    found_match = True
                    break # Exit the 'end_offset' loop
            
            if found_match:
                break # Exit the 'start_attempt' loop
        
        if not found_match:
            print(f"Warning: Could not find any valid snippet for target length {target_len} in '{source_name}'.")

        if i > 0 and i % 500 == 0:
            print(f"  ...found {len(matches)} flexible matches from '{source_name}'.")

    return matches

# --- 1. Get Target Lengths (same as before) ---
try:
    target_lengths = [len(quote.split()) for quote in final_sample_df['QUOTE'].astype(str)]
    print(f"Acquired {len(target_lengths)} target lengths from 'final_sample_df'.")
except NameError:
    print("Error: `final_sample_df` not found. Please run the previous cell first.")
    target_lengths = []

if target_lengths:
    # --- 2. Split Target Lengths (same as before) ---
    num_brown = int(len(target_lengths) * 0.4)
    brown_target_lengths = target_lengths[:num_brown]
    wiki_target_lengths = target_lengths[num_brown:]
    print(f"Assigned {len(brown_target_lengths)} lengths to Brown and {len(wiki_target_lengths)} to Wikipedia.")

    # --- 3. Gather Source TEXT (same as before) ---
    print("\nLoading and preparing source texts...")
    non_fiction_categories = ['news', 'editorial', 'reviews', 'religion', 'hobbies', 'lore', 'belles_lettres', 'government', 'learned', 'reportage']
    brown_words = list(nltk.corpus.brown.words(categories=non_fiction_categories))
    print(f"Loaded {len(brown_words)} words from Brown (non-fiction).")

    print("Streaming Wikipedia documents...")
    wiki_data = load_dataset("wikimedia/wikipedia", "20231101.en", split='train', streaming=True)
    total_wiki_words_needed = sum(wiki_target_lengths) * 4 # Increased buffer for flexible search
    wiki_text_stream = ""
    for doc in wiki_data:
        # A simple estimate to avoid processing the whole 90GB dump
        if len(wiki_text_stream) > total_wiki_words_needed * 6: 
            break
        wiki_text_stream += doc['text'] + " "
    wiki_words = wiki_text_stream.split()
    print(f"Loaded {len(wiki_words)} words from Wikipedia.")

    # --- 4. Find Flexible Contiguous Matches ---
    print("\nFinding flexible, context-aware matches from Brown...")
    final_brown_matches = find_flexible_contiguous_matches(brown_words, brown_target_lengths, "Brown", tolerance=LENGTH_TOLERANCE)

    print("\nFinding flexible, context-aware matches from Wikipedia...")
    final_wiki_matches = find_flexible_contiguous_matches(wiki_words, wiki_target_lengths, "Wikipedia", tolerance=LENGTH_TOLERANCE)
    
    # --- 5. Combine and Create Final DataFrame ---
    print("\nCombining corpora into the final baseline DataFrame...")
    brown_df_part = pd.DataFrame({'text': final_brown_matches, 'source': 'brown_non_fiction'})
    wiki_df_part = pd.DataFrame({'text': final_wiki_matches, 'source': 'wikipedia'})
    
    # Concatenate in order to preserve the row-for-row length correspondence
    final_baseline_df = pd.concat([brown_df_part, wiki_df_part], ignore_index=True)

    # --- 6. Display Results ---
    print("\nFlexible, context-aware, length-matched baseline corpus creation complete.")
    display(final_baseline_df.head())
    print("\nCorpus composition:")
    print(final_baseline_df['source'].value_counts())
    print(f"\nFinal DataFrame shape: {final_baseline_df.shape}")

NLTK data (Brown corpus) already available.
Acquired 5245 target lengths from 'final_sample_df'.
Assigned 2098 lengths to Brown and 3147 to Wikipedia.

Loading and preparing source texts...
Loaded 860006 words from Brown (non-fiction).
Streaming Wikipedia documents...


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Loaded 601766 words from Wikipedia.

Finding flexible, context-aware matches from Brown...
  ...found 501 flexible matches from 'Brown'.
  ...found 1001 flexible matches from 'Brown'.
  ...found 1494 flexible matches from 'Brown'.
  ...found 1993 flexible matches from 'Brown'.

Finding flexible, context-aware matches from Wikipedia...
  ...found 499 flexible matches from 'Wikipedia'.
  ...found 998 flexible matches from 'Wikipedia'.
  ...found 1496 flexible matches from 'Wikipedia'.
  ...found 1996 flexible matches from 'Wikipedia'.
  ...found 2494 flexible matches from 'Wikipedia'.
  ...found 2988 flexible matches from 'Wikipedia'.

Combining corpora into the final baseline DataFrame...

Flexible, context-aware, length-matched baseline corpus creation complete.


,text,source
0,The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place .,brown_non_fiction
1,"City Executive Committee , which had over-all charge of the election , `` deserves the praise and thanks of the City of Atlanta '' for the manner in which the election was conducted . The September-October term jury had been charged by Fulton Superior Court Judge Durwood Pye to investigate repor...",brown_non_fiction
2,Georgia's registration and election laws `` are outmoded or inadequate and often ambiguous '' .,brown_non_fiction
3,"The grand jury commented on a number of other topics , among them the Atlanta and Fulton County purchasing departments which it said `` are well operated and follow generally accepted practices which inure to the best interest of both governments '' .",brown_non_fiction
4,State Welfare Department's handling of federal funds granted for child welfare services in foster homes .,brown_non_fiction



Corpus composition:
source
wikipedia            3129
brown_non_fiction    2090
Name: count, dtype: int64

Final DataFrame shape: (5219, 2)


In [62]:
%pip install langdetect

python(8957) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



[notice] A new release of pip is available: 23.2.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [63]:
import pandas as pd
import numpy as np
import re
from langdetect import detect, LangDetectException

# --- Pre-computation Check ---
# Verify that all required DataFrames exist before proceeding.
required_dfs = ['final_sample_df', 't50_df', 'top_6k_deduplicated_df', 'final_baseline_df']
for df_name in required_dfs:
    if df_name not in locals():
        raise NameError(f"DataFrame '{df_name}' not found. Please ensure the previous cells have been run successfully.")

# --- 1. Reset Indexes for Safe Concatenation ---
final_sample_df.reset_index(drop=True, inplace=True)
t50_df.reset_index(drop=True, inplace=True)
top_6k_deduplicated_df.reset_index(drop=True, inplace=True)
final_baseline_df.reset_index(drop=True, inplace=True)


# --- 2. Prepare and Rename Columns (Corrected) ---
df_google = final_sample_df.copy()
df_google.rename(columns={
    'QUOTE': 'Goodreads Sample Quote', 
    'LIKES': 'Likes of Sample Quote',
    'AUTHOR': 'Author of Goodreads Sample Book',
    'TITLE': 'Title of Goodreads Sample Book', 
    'language_or_country': 'Language of Author of Goodreads Sample Book',
    'publication_date': 'Publication Date of Goodreads Sample Book',
    'genres': 'Genre of Goodreads Sample Book'
}, inplace=True)

df_t50 = t50_df.copy()
df_t50.rename(columns={'T50 quote': 'T50 Quote', 'T50 full context': 'T50 Full Context', 'T50 quote-free context': 'T50 Quote-Free Context', 'T50 quote-free context length matched': 'T50 Quote-Free Context Length Matched'}, inplace=True)

df_popular = top_6k_deduplicated_df.copy()
# Note: top_6k_deduplicated_df already has properly renamed columns from previous cell

df_baseline = final_baseline_df.copy()
df_baseline.rename(columns={'text': 'Non-Literary Baseline'}, inplace=True)

print("Columns renamed successfully.")
print("-" * 30)


# --- 3. Concatenate DataFrames Side-by-Side ---
all_text_df = pd.concat([
    df_google[['Goodreads Sample Quote', 'Google Books Length Matched Snippet', 'Google Books Page Text']],
    df_t50[['T50 Quote', 'T50 Full Context', 'T50 Quote-Free Context', 'T50 Quote-Free Context Length Matched']],
    df_baseline[['Non-Literary Baseline']],
    df_popular[['Goodreads Popular Quote']]
], axis=1)

final_column_order = [
    'Goodreads Sample Quote', 'Google Books Length Matched Snippet', 'T50 Quote',
    'T50 Quote-Free Context Length Matched', 'Non-Literary Baseline', 'Goodreads Popular Quote', 
    'Google Books Page Text', 'T50 Full Context', 'T50 Quote-Free Context'
]
final_merged_df = all_text_df[final_column_order]

metadata_df = df_google.drop(columns=[
    'Goodreads Sample Quote', 'Google Books Length Matched Snippet', 'Google Books Page Text',
    'quote_length', 'matched_snippet_length'
], errors='ignore')

# Create the metadata section with specific column order: Likes of Sample Quote, Likes of Popular Quote, then other metadata
likes_sample_df = metadata_df[['Likes of Sample Quote']]
likes_popular_df = df_popular[['Likes of Popular Quote']]
other_metadata_df = metadata_df.drop(columns=['Likes of Sample Quote'], errors='ignore')

final_merged_df_with_metadata = pd.concat([final_merged_df, likes_sample_df, likes_popular_df, other_metadata_df], axis=1)

print("DataFrames merged successfully.")
print("-" * 30)


# --- 4. Data Cleaning and Filtering ---
text_columns_to_clean = final_column_order

print("Applying whitespace and chapter normalization...")
for col in text_columns_to_clean:
    if col in final_merged_df_with_metadata.columns:
        final_merged_df_with_metadata[col] = final_merged_df_with_metadata[col].astype(str).replace('nan', '')
        final_merged_df_with_metadata[col] = final_merged_df_with_metadata[col].str.replace(r'\s+', ' ', regex=True).str.strip()
        chapter_pattern = r'^\s*CHAPTER\s+[IVXLCDM\d]+\s*$'
        final_merged_df_with_metadata[col] = final_merged_df_with_metadata[col].str.replace(chapter_pattern, '', regex=True, flags=re.IGNORECASE).str.strip()

print("Normalization complete.")

print("\nFiltering for English-language quotes in 'Goodreads Sample Quote'...")
initial_rows = len(final_merged_df_with_metadata)
def is_english(text):
    if not isinstance(text, str) or len(text.split()) < 3: return False
    try: return detect(text) == 'en'
    except LangDetectException: return False

english_mask = final_merged_df_with_metadata['Goodreads Sample Quote'].apply(is_english)
final_merged_df_with_metadata = final_merged_df_with_metadata[english_mask].copy()
print(f"Removed {initial_rows - len(final_merged_df_with_metadata)} non-English or ambiguous rows.")
print("-" * 30)


Columns renamed successfully.
------------------------------
DataFrames merged successfully.
------------------------------
Applying whitespace and chapter normalization...
Normalization complete.

Filtering for English-language quotes in 'Goodreads Sample Quote'...
Removed 766 non-English or ambiguous rows.
------------------------------


In [64]:
# --- 4.5. Count CHAPTER Instances Before Final Sampling ---
print("Counting instances of 'CHAPTER' patterns in each column...")
print("=" * 90)

# Define the columns to check
text_columns = [
    'Goodreads Sample Quote', 'Google Books Length Matched Snippet', 'T50 Quote',
    'Non-Literary Baseline', 'Goodreads Popular Quote', 'Google Books Page Text', 'T50 Full Context', 'T50 Quote-Free Context', 'T50 Quote-Free Context Length Matched'
]

# Count different CHAPTER patterns
chapter_only_counts = {}
chapter_with_number_counts = {}
chapter_with_roman_counts = {}
total_chapter_only = 0
total_chapter_with_number = 0
total_chapter_with_roman = 0

print(f"{'Column Name':35} {'CHAPTER':>8} {'CHAP NUM':>9} {'CHAP ROM':>10} {'Total':>8}")
print("-" * 90)

for col in text_columns:
    if col in final_merged_df_with_metadata.columns:
        # Convert to string
        col_data = final_merged_df_with_metadata[col].astype(str)
        
        # Count "CHAPTER" followed by Arabic numbers (like CHAPTER 1, CHAPTER 12, etc.)
        chapter_with_number_count = col_data.str.count(r'\bCHAPTER\s+\d+\b').sum()
        
        # Count "CHAPTER" followed by Roman numerals (like CHAPTER I, CHAPTER XIV, CHAPTER XXXVIII, etc.)
        chapter_with_roman_count = col_data.str.count(r'\bCHAPTER\s+[IVXLCDM]+\b').sum()
        
        # Count total "CHAPTER" occurrences
        total_chapter_count = col_data.str.count(r'\bCHAPTER\b').sum()
        
        # Count standalone "CHAPTER" (not followed by numbers or roman numerals)
        chapter_only_count = total_chapter_count - chapter_with_number_count - chapter_with_roman_count
        
        chapter_only_counts[col] = chapter_only_count
        chapter_with_number_counts[col] = chapter_with_number_count
        chapter_with_roman_counts[col] = chapter_with_roman_count
        total_chapter_only += chapter_only_count
        total_chapter_with_number += chapter_with_number_count
        total_chapter_with_roman += chapter_with_roman_count
        
        total_col = chapter_only_count + chapter_with_number_count + chapter_with_roman_count
        print(f"{col:35} {chapter_only_count:>8} {chapter_with_number_count:>9} {chapter_with_roman_count:>10} {total_col:>8}")
    else:
        chapter_only_counts[col] = 0
        chapter_with_number_counts[col] = 0
        chapter_with_roman_counts[col] = 0
        print(f"{col:35} {'N/A':>8} {'N/A':>9} {'N/A':>10} {'N/A':>8}")

print("-" * 90)
grand_total = total_chapter_only + total_chapter_with_number + total_chapter_with_roman
print(f"{'TOTAL ACROSS ALL COLUMNS':35} {total_chapter_only:>8} {total_chapter_with_number:>9} {total_chapter_with_roman:>10} {grand_total:>8}")
print("=" * 90)

# Show some examples of all three patterns
print("\nSample examples of CHAPTER patterns:")
print("-" * 60)

for col in ['T50 Full Context', 'Goodreads Popular Quote', 'T50 Quote-Free Context', 'T50 Quote-Free Context Length Matched', 'Google Books Page Text', 'T50 Quote', 'Non-Literary Baseline', 'Goodreads Sample Quote', 'Google Books Length Matched Snippet']:  # Check the columns that had instances
    if col in final_merged_df_with_metadata.columns:
        col_data = final_merged_df_with_metadata[col].astype(str)
        
        # Find examples of "CHAPTER NUM" (Arabic numbers) pattern
        chapter_num_rows = col_data[col_data.str.contains(r'\bCHAPTER\s+\d+\b', na=False)]
        if len(chapter_num_rows) > 0:
            print(f"\n{col} - CHAPTER [Arabic Numbers] examples:")
            for i, (idx, text) in enumerate(chapter_num_rows.head(3).items()):
                # Extract the CHAPTER NUM part specifically
                import re
                match = re.search(r'\bCHAPTER\s+\d+\b', text)
                if match:
                    chapter_part = match.group()
                    # Show some context around it
                    start_pos = max(0, match.start() - 20)
                    end_pos = min(len(text), match.end() + 50)
                    context = text[start_pos:end_pos].strip()
                    print(f"  Row {idx}: ...{context}...")
            if len(chapter_num_rows) > 3:
                print(f"  ... and {len(chapter_num_rows) - 3} more instances")
        
        # Find examples of "CHAPTER XXX" (Roman numerals) pattern
        chapter_roman_rows = col_data[col_data.str.contains(r'\bCHAPTER\s+[IVXLCDM]+\b', na=False)]
        if len(chapter_roman_rows) > 0:
            print(f"\n{col} - CHAPTER [Roman Numerals] examples:")
            for i, (idx, text) in enumerate(chapter_roman_rows.head(3).items()):
                # Extract the CHAPTER XXX part specifically
                import re
                match = re.search(r'\bCHAPTER\s+[IVXLCDM]+\b', text)
                if match:
                    chapter_part = match.group()
                    # Show some context around it
                    start_pos = max(0, match.start() - 20)
                    end_pos = min(len(text), match.end() + 50)
                    context = text[start_pos:end_pos].strip()
                    print(f"  Row {idx}: ...{context}...")
            if len(chapter_roman_rows) > 3:
                print(f"  ... and {len(chapter_roman_rows) - 3} more instances")
        
        # Find examples of standalone "CHAPTER" (that are NOT followed by numbers or romans)
        chapter_only_rows = col_data[
            col_data.str.contains(r'\bCHAPTER\b', na=False) & 
            ~col_data.str.contains(r'\bCHAPTER\s+\d+\b', na=False) &
            ~col_data.str.contains(r'\bCHAPTER\s+[IVXLCDM]+\b', na=False)
        ]
        if len(chapter_only_rows) > 0:
            print(f"\n{col} - standalone CHAPTER examples:")
            for i, (idx, text) in enumerate(chapter_only_rows.head(2).items()):
                # Show context around CHAPTER
                import re
                match = re.search(r'\bCHAPTER\b', text)
                if match:
                    start_pos = max(0, match.start() - 20)
                    end_pos = min(len(text), match.end() + 50)
                    context = text[start_pos:end_pos].strip()
                    print(f"  Row {idx}: ...{context}...")
            if len(chapter_only_rows) > 2:
                print(f"  ... and {len(chapter_only_rows) - 2} more instances")

print("\n" + "=" * 90)

# --- 4.6. Remove CHAPTER Instances and Clean Text ---
print("\nRemoving CHAPTER patterns and cleaning text...")
print("-" * 60)

import re

def clean_chapter_patterns(text):
    """
    Remove various CHAPTER patterns and clean up spacing issues.
    """
    if not isinstance(text, str):
        return text
    
    # Store original text for comparison
    original_text = text
    
    # Remove CHAPTER patterns in order of specificity (most specific first)
    
    # 1. Remove "CHAPTER" followed by Roman numerals (e.g., CHAPTER XXXVIII, CHAPTER V)
    text = re.sub(r'\bCHAPTER\s+[IVXLCDM]+\b[^\w]*', '', text, flags=re.IGNORECASE)
    
    # 2. Remove "CHAPTER" followed by Arabic numbers (e.g., CHAPTER 14, CHAPTER 5)
    text = re.sub(r'\bCHAPTER\s+\d+\b[^\w]*', '', text, flags=re.IGNORECASE)
    
    # 3. Remove standalone "CHAPTER" with any following punctuation/symbols
    # This pattern looks for CHAPTER followed by non-word characters (punctuation, symbols, etc.)
    text = re.sub(r'\bCHAPTER\b[^\w]*', '', text, flags=re.IGNORECASE)
    
    # Fix spacing issues that result from removals:
    
    # 1. Replace multiple consecutive spaces with single space
    text = re.sub(r'\s+', ' ', text)
    
    # 2. Remove spaces before punctuation
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)
    
    # 3. Ensure proper spacing after punctuation
    text = re.sub(r'([,.;:!?])(\w)', r'\1 \2', text)
    
    # 4. Remove leading/trailing whitespace
    text = text.strip()
    
    # 5. Fix cases where removal left sentences starting with lowercase
    # Split into sentences and capitalize first letter of each
    sentences = re.split(r'([.!?]+\s*)', text)
    cleaned_sentences = []
    
    for i, sentence in enumerate(sentences):
        if i % 2 == 0:  # Even indices are actual sentences, odd indices are delimiters
            sentence = sentence.strip()
            if sentence and sentence[0].islower():
                sentence = sentence[0].upper() + sentence[1:]
            cleaned_sentences.append(sentence)
        else:
            cleaned_sentences.append(sentence)
    
    text = ''.join(cleaned_sentences)
    
    # 6. Final cleanup: remove any remaining double spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning to all text columns
total_cleaned = 0
for col in text_columns:
    if col in final_merged_df_with_metadata.columns:
        # Count how many rows will be affected
        col_data = final_merged_df_with_metadata[col].astype(str)
        affected_rows = col_data.str.contains(r'\bCHAPTER\b', na=False).sum()
        
        if affected_rows > 0:
            print(f"Cleaning {affected_rows} rows in column '{col}'...")
            
            # Apply the cleaning function
            final_merged_df_with_metadata[col] = final_merged_df_with_metadata[col].apply(clean_chapter_patterns)
            total_cleaned += affected_rows

print(f"\nCleaning complete. Processed {total_cleaned} total instances across all columns.")

# --- 4.7. Verify Removal ---
print("\n" + "=" * 90)
print("Verifying CHAPTER pattern removal...")
print("=" * 90)

# Re-count to verify removal
verification_totals = {}
for col in text_columns:
    if col in final_merged_df_with_metadata.columns:
        col_data = final_merged_df_with_metadata[col].astype(str)
        remaining_chapters = col_data.str.count(r'\bCHAPTER\b').sum()
        verification_totals[col] = remaining_chapters
        print(f"{col:35} {remaining_chapters:>8}")
    else:
        verification_totals[col] = 0
        print(f"{col:35} {'N/A':>8}")

total_remaining = sum(verification_totals.values())
print("-" * 90)
print(f"{'TOTAL REMAINING CHAPTERS':35} {total_remaining:>8}")

if total_remaining == 0:
    print("SUCCESS: All CHAPTER patterns have been successfully removed!")
else:
    print(f"WARNING: {total_remaining} CHAPTER patterns still remain. Manual inspection may be needed.")

print("=" * 90)

Counting instances of 'CHAPTER' patterns in each column...
Column Name                          CHAPTER  CHAP NUM   CHAP ROM    Total
------------------------------------------------------------------------------------------
Goodreads Sample Quote                     0         0          0        0
Google Books Length Matched Snippet        0         0          0        0
T50 Quote                                  0         0          0        0
Non-Literary Baseline                      0         0          0        0
Goodreads Popular Quote                    1         0          0        1
Google Books Page Text                     0         0          0        0
T50 Full Context                          31        96       1381     1508
T50 Quote-Free Context                    25        96       1365     1486
T50 Quote-Free Context Length Matched        4        25        131      160
------------------------------------------------------------------------------------------
TOTAL A

In [65]:
# --- 4.8. Whitespace Normalization and Deduplication ---
print("Starting whitespace normalization and deduplication...")
print("=" * 70)

# Get the text columns for normalization
text_columns = [
    'Goodreads Sample Quote', 'Google Books Length Matched Snippet', 'T50 Quote',
    'Non-Literary Baseline', 'Goodreads Popular Quote', 'Google Books Page Text', 'T50 Full Context', 'T50 Quote-Free Context', 'T50 Quote-Free Context Length Matched'
]

# Step 1: Advanced whitespace normalization
print("Step 1: Performing advanced whitespace normalization...")
print("-" * 50)

import re

def normalize_whitespace(text):
    """
    Comprehensive whitespace normalization function.
    """
    if not isinstance(text, str) or text == 'nan':
        return ''
    
    # Remove null bytes and other problematic characters
    text = text.replace('\x00', '').replace('\ufeff', '')
    
    # Normalize different types of spaces and whitespace characters
    # Replace various unicode space characters with regular space
    text = re.sub(r'[\u00A0\u1680\u2000-\u200B\u202F\u205F\u3000\uFEFF]', ' ', text)
    
    # Replace tabs and newlines with spaces
    text = re.sub(r'[\t\n\r\f\v]', ' ', text)
    
    # Replace multiple consecutive spaces with single space
    text = re.sub(r' +', ' ', text)
    
    # Strip leading and trailing whitespace
    text = text.strip()
    
    return text

# Apply normalization to all text columns
for col in text_columns:
    if col in final_merged_df_with_metadata.columns:
        print(f"  Normalizing whitespace in '{col}'...")
        final_merged_df_with_metadata[col] = final_merged_df_with_metadata[col].apply(normalize_whitespace)

print("✓ Whitespace normalization complete.")

# Step 2: Check for duplicates in each column
print("\nStep 2: Checking for duplicates in each column...")
print("-" * 50)

duplicate_counts = {}
total_duplicates_found = 0

for col in text_columns:
    if col in final_merged_df_with_metadata.columns:
        # Count duplicates (exclude empty strings from duplicate counting)
        col_data = final_merged_df_with_metadata[col]
        non_empty_data = col_data[col_data != '']
        
        if len(non_empty_data) > 0:
            duplicates = non_empty_data.duplicated().sum()
            duplicate_counts[col] = duplicates
            total_duplicates_found += duplicates
            
            if duplicates > 0:
                print(f"  !  '{col}': {duplicates} duplicate entries found")
                
                # Show examples of duplicates
                duplicate_values = non_empty_data[non_empty_data.duplicated(keep=False)].value_counts()
                print(f"      Top duplicated values:")
                for i, (value, count) in enumerate(duplicate_values.head(3).items()):
                    preview = value[:100] + "..." if len(value) > 100 else value
                    print(f"        '{preview}' appears {count} times")
            else:
                print(f"  ✓ '{col}': No duplicates found")
        else:
            duplicate_counts[col] = 0
            print(f"  ✓ '{col}': No non-empty data to check")

print(f"\nTotal duplicate entries found across all columns: {total_duplicates_found}")

# Step 3: Remove duplicate rows based on each text column
print("\nStep 3: Removing duplicate rows...")
print("-" * 50)

initial_row_count = len(final_merged_df_with_metadata)
rows_removed_by_column = {}

for col in text_columns:
    if col in final_merged_df_with_metadata.columns and duplicate_counts.get(col, 0) > 0:
        before_count = len(final_merged_df_with_metadata)
        
        # Remove duplicates in this column (keep first occurrence)
        # Only consider non-empty values for deduplication
        mask_non_empty = final_merged_df_with_metadata[col] != ''
        duplicated_mask = final_merged_df_with_metadata[col].duplicated(keep='first') & mask_non_empty
        
        final_merged_df_with_metadata = final_merged_df_with_metadata[~duplicated_mask].copy()
        
        after_count = len(final_merged_df_with_metadata)
        rows_removed = before_count - after_count
        rows_removed_by_column[col] = rows_removed
        
        if rows_removed > 0:
            print(f"  Removed {rows_removed} rows due to duplicates in '{col}'")
        else:
            print(f"  No rows removed for '{col}'")

total_rows_removed = initial_row_count - len(final_merged_df_with_metadata)
print(f"\nDeduplication complete!")
print(f"Original rows: {initial_row_count}")
print(f"Final rows: {len(final_merged_df_with_metadata)}")
print(f"Total rows removed: {total_rows_removed}")

# Step 4: Final verification - check for any remaining duplicates
print("\nStep 4: Final verification...")
print("-" * 50)

final_duplicate_check = {}
any_remaining_duplicates = False

for col in text_columns:
    if col in final_merged_df_with_metadata.columns:
        col_data = final_merged_df_with_metadata[col]
        non_empty_data = col_data[col_data != '']
        
        if len(non_empty_data) > 0:
            remaining_duplicates = non_empty_data.duplicated().sum()
            final_duplicate_check[col] = remaining_duplicates
            
            if remaining_duplicates > 0:
                print(f" !'{col}': {remaining_duplicates} duplicates still remain")
                any_remaining_duplicates = True
            else:
                print(f"  ✓ '{col}': Clean, no duplicates")
        else:
            final_duplicate_check[col] = 0
            print(f"  ✓ '{col}': No non-empty data")

if not any_remaining_duplicates:
    print("\nSUCCESS: All columns are now free of duplicates!")
else:
    print("\nWARNING: Some duplicates may still remain. Manual review recommended.")

print("=" * 70)
print(f"Dataset ready for final sampling: {len(final_merged_df_with_metadata)} rows")


Starting whitespace normalization and deduplication...
Step 1: Performing advanced whitespace normalization...
--------------------------------------------------
  Normalizing whitespace in 'Goodreads Sample Quote'...
  Normalizing whitespace in 'Google Books Length Matched Snippet'...
  Normalizing whitespace in 'T50 Quote'...
  Normalizing whitespace in 'Non-Literary Baseline'...
  Normalizing whitespace in 'Goodreads Popular Quote'...
  Normalizing whitespace in 'Google Books Page Text'...
  Normalizing whitespace in 'T50 Full Context'...
  Normalizing whitespace in 'T50 Quote-Free Context'...
  Normalizing whitespace in 'T50 Quote-Free Context Length Matched'...
✓ Whitespace normalization complete.

Step 2: Checking for duplicates in each column...
--------------------------------------------------
  !  'Goodreads Sample Quote': 89 duplicate entries found
      Top duplicated values:
        'What strange phenomena we find in a great city, all we need do is stroll about with our ey

In [66]:
# ═══════════ TEXT NORMALIZATION FOR CREATIVITY METRICS - 1 ═══════════
# Enhanced normalization specifically designed for computational analysis

import re
import unicodedata
import numpy as np

print("Starting comprehensive text normalization for all text columns...")
print("Enhanced for creativity metrics - aggressive punctuation normalization")
print("=" * 70)

# ── 1. ENHANCED NORMALISATION FUNCTION ────────────────────────────────────────
_soft = "\u00AD"

# More comprehensive regex patterns
rx_ctrl = re.compile(r"[\u0000-\u001F\u007F-\u009F]")  # all control chars
rx_ws = re.compile(r"\s+")                             # collapse whitespace
rx_sep = re.compile(r"\s+([.,?!;:])")                  # space before punct

# COMPREHENSIVE ellipses patterns (all unicode variants + multiple dots)
rx_ell = re.compile(r"(\.{2,}|…|‥|⋯|‰|‱|․|‸|⁖|⁘|⁙|⁛)")

# All dash variants 
rx_dash = re.compile(r"\s*(?:--+|—+|–+|−+|⸺+|⸻+)\s*")

# Footnote markers
rx_caret = re.compile(r"\^\d+")
rx_brack = re.compile(r"\[\d+\]")

# COMPREHENSIVE smart quotes mapping (all unicode variants)
smart_q = str.maketrans({
    # Standard smart quotes
    "\u201c": '"', "\u201d": '"', "\u2018": "'", "\u2019": "'",
    # Additional quote variants
    "\u201a": "'", "\u201e": '"', "\u201f": '"', "\u2039": "'", "\u203a": "'",
    "\u00ab": '"', "\u00bb": '"', "\u02ba": '"', "\u02ee": '"',
    # More exotic quote variants found in digitized texts
    "\u275d": '"', "\u275e": '"', "\u276e": "'", "\u276f": "'",
    "\u2760": '"', "\u2761": '"', "\u275b": "'", "\u275c": "'",
    "\u275f": '"', "\u2762": '"', "\u2763": '"',
    # Modifier quotes
    "\u02bb": "'", "\u02bc": "'", "\u02bd": "'", "\u02be": "'", "\u02bf": "'",
    "\u02c0": "'", "\u02c1": "'", "\u02c2": "'", "\u02c3": "'", "\u02c4": "'",
    "\u02c5": "'", "\u02c6": "'", "\u02c7": "'", "\u02c8": "'",
    # Fullwidth quotes (common in some digitized texts)
    "\uff02": '"', "\uff07": "'",
})

# Superscripts and subscripts
supers = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789")
subs = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")

def clean_text(t: str) -> str:
    """
    Enhanced text cleaning for creativity metrics:
    - Unicode normalization (NFC)
    - ALL smart quote variants to regular quotes
    - All superscript/subscript numbers to regular numbers
    - Removes soft hyphens and ALL control characters
    - Standardizes ALL ellipses variants to single …
    - Standardizes ALL dash variants to single —
    - Removes footnote markers
    - Aggressive whitespace normalization
    """
    if not isinstance(t, str):
        return ""
    
    # Unicode normalization
    t = unicodedata.normalize("NFC", t)
    
    # Character translations
    t = t.translate(smart_q).translate(supers).translate(subs)
    
    # Remove problematic characters
    t = t.replace(_soft, "")  # soft hyphens
    t = rx_ctrl.sub("", t)    # control characters
    
    # Normalize punctuation patterns
    t = rx_ell.sub(" … ", t)    # all ellipses to spaced single …
    t = rx_dash.sub(" — ", t)   # all dashes to spaced single —
    
    # Remove footnotes
    t = rx_caret.sub("", t)   # ^1, ^2, etc.
    t = rx_brack.sub("", t)   # [1], [2], etc.
    
    # Enhanced whitespace cleanup
    t = rx_ws.sub(" ", t)     # collapse multiple spaces first
    t = rx_sep.sub(r"\1", t)  # remove spaces before punctuation
    
    # Additional cleanup for multiple spaces
    t = re.sub(r'  +', ' ', t)  # catch any remaining multiple spaces
    t = re.sub(r'[\u00A0\u1680\u2000-\u200B\u202F\u205F\u3000\uFEFF]+', ' ', t)  # unicode spaces
    t = re.sub(r' +', ' ', t)   # final multiple space cleanup
    
    # FINAL ellipses cleanup (aggressive - catch any missed patterns)
    t = re.sub(r'\.{2,}', ' … ', t)  # any 2+ dots to spaced ellipsis
    t = re.sub(r'[‥⋯‰‱․‸⁖⁘⁙⁛]', ' … ', t)  # any unicode ellipses variants
    t = re.sub(r' +', ' ', t)   # clean up spaces again after ellipses fixes
    
    return t.strip()

# ── 2. APPLY NORMALIZATION TO ALL TEXT COLUMNS ────────────────────────────────
# Define all text columns that need normalization
text_columns_to_normalize = [
    'Goodreads Sample Quote', 
    'Google Books Length Matched Snippet', 
    'T50 Quote',
    'Non-Literary Baseline', 
    'Goodreads Popular Quote', 
    'Google Books Page Text', 
    'T50 Full Context',
    'T50 Quote-Free Context',
    'T50 Quote-Free Context Length Matched'
]

print("Applying comprehensive text normalization to each column...")
print("-" * 50)

# Track normalization statistics
normalization_stats = {}
total_changed = 0

for col in text_columns_to_normalize:
    if col in final_merged_df_with_metadata.columns:
        print(f"  Processing '{col}'...")
        
        # Get the original data
        original_data = final_merged_df_with_metadata[col].fillna("").astype(str)
        
        # Apply normalization
        normalized_data = original_data.apply(clean_text)
        
        # Count how many entries were changed
        changes = (original_data != normalized_data).sum()
        normalization_stats[col] = changes
        total_changed += changes
        
        # Update the dataframe
        final_merged_df_with_metadata[col] = normalized_data
        
        print(f"    SUCCESS: Normalized {changes} entries in '{col}'")
        
        # Show a sample of changes if any occurred
        if changes > 0:
            changed_mask = original_data != normalized_data
            changed_indices = original_data[changed_mask].index[:2]  # Show first 2 examples
            
            for idx in changed_indices:
                orig = original_data.iloc[idx]
                norm = normalized_data.iloc[idx]
                
                # Show a preview of the change
                orig_preview = orig[:100] + "..." if len(orig) > 100 else orig
                norm_preview = norm[:100] + "..." if len(norm) > 100 else norm
                
                print(f"      Example change (row {idx}):")
                print(f"        Before: {repr(orig_preview)}")
                print(f"        After:  {repr(norm_preview)}")
                break  # Just show one example per column
    else:
        print(f"  ⚠ Column '{col}' not found in dataset")
        normalization_stats[col] = 0

print("\n" + "-" * 50)
print("Normalization Summary:")
for col, changes in normalization_stats.items():
    if col in final_merged_df_with_metadata.columns:
        print(f"  {col:35} {changes:>6} entries normalized")

print(f"\nTotal entries normalized across all columns: {total_changed}")

# ── 3. VERIFICATION ─────────────────────────────────────────────────────────────
print("\nVerifying normalization results...")
print("-" * 50)

# Check for common OCR/formatting issues that should now be cleaned
verification_patterns = {
    'unnormalized_smart_quotes': r'[\u201c\u201d\u2018\u2019\u201a\u201e\u2039\u203a\u00ab\u00bb\u02ba\u02ee\u201f]',
    'superscripts': r'[⁰¹²³⁴⁵⁶⁷⁸⁹]',
    'soft_hyphens': _soft,
    'control_chars': r'[\u0000-\u001F]',
    'multiple_spaces': r'  +',
    'footnote_markers': r'(\^\d+|\[\d+\])',
    'unnormalized_multiple_dashes': r'--+',
    'unnormalized_en_dash': r'–+',
    'unnormalized_ellipses': r'\.{2,}|[‥⋯‰‱․‸⁖⁘⁙⁛]'
}

remaining_issues = {}
total_remaining = 0

for pattern_name, pattern in verification_patterns.items():
    count = 0
    for col in text_columns_to_normalize:
        if col in final_merged_df_with_metadata.columns:
            col_data = final_merged_df_with_metadata[col].astype(str)
            count += col_data.str.count(pattern).sum()
    
    remaining_issues[pattern_name] = count
    total_remaining += count
    
    if count > 0:
        print(f"  WARNING {pattern_name}: {count} instances still found")
    else:
        print(f"  SUCCESS {pattern_name}: Successfully normalized")

if total_remaining == 0:
    print(f"\nSUCCESS: All text normalization completed successfully!")
    print(f"   No remaining formatting issues detected.")
else:
    print(f"\nPARTIAL SUCCESS: {total_remaining} formatting issues remain.")
    print(f"   Most content has been normalized, remaining issues may need manual review.")

print("=" * 70)
print("Text normalization complete. Data ready for final processing.")


Starting comprehensive text normalization for all text columns...
Enhanced for creativity metrics - aggressive punctuation normalization
Applying comprehensive text normalization to each column...
--------------------------------------------------
  Processing 'Goodreads Sample Quote'...
    SUCCESS: Normalized 885 entries in 'Goodreads Sample Quote'
      Example change (row 0):
        Before: 'Houses, I have come to believe, like love, like nature herself, should not reassure, should not atte...'
        After:  'Houses, I have come to believe, like love, like nature herself, should not reassure, should not atte...'
  Processing 'Google Books Length Matched Snippet'...
    SUCCESS: Normalized 337 entries in 'Google Books Length Matched Snippet'
      Example change (row 15):
        Before: 'Wild animals are terrified of fire. The flames are leaping higher and the wind is carrying the smoke...'
        After:  'Wild animals are terrified of fire. The flames are leaping higher and th

In [67]:
# ═══════════ TEXT NORMALIZATION FOR CREATIVITY METRICS - 2 ═══════════
# Enhanced normalization specifically designed for computational analysis

import re
import pandas as pd

# Define the text columns to be cleaned from your main dataframe
columns_to_normalize = [
    "Goodreads Sample Quote",
    "Google Books Length Matched Snippet",
    "Non-Literary Baseline",
    "Goodreads Popular Quote",
    "T50 Quote",
    "T50 Quote-Free Context",
    "T50 Quote-Free Context Length Matched",
    "T50 Full Context",
    "Google Books Page Text"
]

# This function applies all the regex cleaning rules
def normalize_text(text):
    if not isinstance(text, str):
        return text

    # Rule 1: Remove mangled stage directions like '_]' at the start of a line.
    text = re.sub(r'^\s*_\s*\]\s*', '', text)

    # Rule 2: Remove underscores used for emphasis (e.g. _word_ -> word).
    # This looks for underscores that are not part of a word (e.g. not in "variable_name").
    text = re.sub(r'\b_([^_]+)_\b', r'\1', text)

    # Rule 3: Clean up content within brackets, like stripping extra space and underscores.
    # e.g., "[ _ Exit. Knocking within. _ ]" -> "[Exit. Knocking within.]"
    text = re.sub(r'\[(.*?)\]', lambda m: '[' + m.group(1).strip(' _') + ']', text)
    text = re.sub(r'\((.*?)\)', lambda m: '(' + m.group(1).strip(' _') + ')', text)
    text = re.sub(r'\{(.*?)\}', lambda m: '{' + m.group(1).strip(' _') + '}', text)
    
    # Rule 4: Re-join split contractions (e.g., "should n't" -> "shouldn't")
    # This also covers cases like "is n't", "are n't", "was n't", etc.
    text = re.sub(r"(\w+)\s+(n't\b)", r"\1n't", text)
    
    # Rule 5: Re-join split possessives (e.g., "tiger 's" -> "tiger's")
    text = re.sub(r"(\w+)\s+('s\b)", r"\1's", text)
    
    # Rule 6: Remove space after an opening bracket (e.g., "( text" -> "(text")
    text = re.sub(r"([(\[{])\s+", r"\1", text)
    
    # Rule 7: Remove space before a closing bracket (e.g., "text )" -> "text)")
    text = re.sub(r"\s+([)\]}])", r"\1", text)

    # Rule 8: I 'm, He 's, etc. to correct form
    text = re.sub(r"(\w+)\s+('(m|re|ve|ll|d)\b)", r"\1\2", text)

    # Rule 9: Remove stray underscores attached to the inside of brackets
    text = re.sub(r'\[\s*_', '[', text)
    text = re.sub(r'_\s*\]', ']', text)

    # Rule 10: Remove empty bracket pairs like [], (), or {}
    text = re.sub(r'\[\s*\]|\(\s*\)|\{\s*\}', '', text)
    
    return text.strip()

# --- Main cleaning and reporting loop ---

# Make a copy to safely track changes
df_to_clean = final_merged_df_with_metadata.copy()

for column in columns_to_normalize:
    if column not in df_to_clean.columns:
        print(f"\n--- Column '{column}' not found. Skipping. ---")
        continue

    print(f"\n--- Normalizing Column: {column} ---")
    
    changed_rows = []
    
    for index, row in df_to_clean.iterrows():
        original_text = row[column]
        cleaned_text = normalize_text(original_text)
        
        # If a change was made, record it and update the dataframe
        if original_text != cleaned_text:
            changed_rows.append({
                "index": index,
                "before": original_text,
                "after": cleaned_text
            })
            # Update the cell in our copied dataframe
            df_to_clean.at[index, column] = cleaned_text

    # --- Reporting for the current column ---
    if not changed_rows:
        print("No lines required changes.")
    else:
        print(f"Found and fixed {len(changed_rows)} lines with normalization issues.")
        print("\nExample Changes (showing up to 5):")
        print("-" * 25)
        for i, change in enumerate(changed_rows[:5]):
            print(f"[{i+1}] BEFORE: {change['before']}")
            print(f"    AFTER:  {change['after']}\n")

# Replace the original dataframe with the cleaned version
final_merged_df_with_metadata = df_to_clean
print("\n═══════════ NORMALIZATION COMPLETE ═══════════")
print("The 'final_merged_df_with_metadata' dataframe has been updated in place.")


--- Normalizing Column: Goodreads Sample Quote ---
Found and fixed 26 lines with normalization issues.

Example Changes (showing up to 5):
-------------------------
[1] BEFORE: You had two prerequisites." Regin plopped down on a snowbank. "And I do believe I have Russian ex-mil contacts, and I speak the language-" "Oh, come on! I've since learned that you do not by any stretch. You think Dostoyevsky is Russian for 'How 's it hanging?'" She blinked up at Kaderin as she paced by. "Then how do you say it?" "I-don't-know." "Then how do you know it's not Dostoyevsky? No. Really." She blew a bubble with her gum — possibly the first to do so at this location — but it flash-froze, and she had to crunch it back to gum consistency with her molars. "Obi-Wan, I was your only hope." (Kaderin and Regin)
    AFTER:  You had two prerequisites." Regin plopped down on a snowbank. "And I do believe I have Russian ex-mil contacts, and I speak the language-" "Oh, come on! I've since learned that you do no

In [68]:
# --- EMPTY CELL ANALYSIS AND CLEANUP ---
print("=" * 80)
print("EMPTY CELL ANALYSIS AND CLEANUP FOR 8 DATA COLUMNS")
print("=" * 80)

# Define the 8 core data columns
data_columns = [
    'Goodreads Sample Quote', 'Google Books Length Matched Snippet', 'T50 Quote',
    'Non-Literary Baseline', 'Goodreads Popular Quote', 'Google Books Page Text', 
    'T50 Full Context', 'T50 Quote-Free Context', 'T50 Quote-Free Context Length Matched'
]

def is_empty_cell(value):
    """Check if a cell is considered empty."""
    if pd.isna(value):
        return True
    str_value = str(value).strip()
    return str_value in ['', 'nan', 'None'] or str_value == ''

def analyze_empty_cells(df, columns):
    """Analyze empty cells in specified columns."""
    total_rows = len(df)
    results = {}
    
    print(f"Total rows: {total_rows:,}\n")
    print(f"{'Column':<35} {'Empty':<8} {'%':<6} {'Filled':<8}")
    print("-" * 60)
    
    # Analyze each column
    for col in columns:
        if col in df.columns:
            empty_count = df[col].apply(is_empty_cell).sum()
            filled_count = total_rows - empty_count
            percentage = (empty_count / total_rows) * 100
            results[col] = empty_count
            print(f"{col:<35} {empty_count:<8,} {percentage:<5.1f}% {filled_count:<8,}")
        else:
            results[col] = 0
            print(f"{col:<35} {'N/A':<8} {'N/A':<6} {'N/A':<8}")
    
    return results

def find_incomplete_rows(df, columns):
    """Find rows with any empty cells in the specified columns."""
    incomplete_mask = pd.Series([False] * len(df), index=df.index)
    
    for col in columns:
        if col in df.columns:
            col_empty_mask = df[col].apply(is_empty_cell)
            incomplete_mask = incomplete_mask | col_empty_mask
    
    return incomplete_mask

# --- Step 1: Analyze Current State ---
print("CURRENT STATE ANALYSIS:")
empty_counts = analyze_empty_cells(final_merged_df_with_metadata, data_columns)

# Find rows with incomplete data
incomplete_rows_mask = find_incomplete_rows(final_merged_df_with_metadata, data_columns)
incomplete_count = incomplete_rows_mask.sum()
complete_count = len(final_merged_df_with_metadata) - incomplete_count

print(f"\n{'SUMMARY':<35} {'Count':<8} {'%'}")
print("-" * 50)
print(f"{'Rows with missing data':<35} {incomplete_count:<8,} {(incomplete_count/len(final_merged_df_with_metadata)*100):<.1f}%")
print(f"{'Complete rows':<35} {complete_count:<8,} {(complete_count/len(final_merged_df_with_metadata)*100):<.1f}%")

# --- Step 2: Perform Deletion ---
print(f"\n" + "=" * 60)
print("REMOVING INCOMPLETE ROWS")
print("=" * 60)

original_count = len(final_merged_df_with_metadata)
print(f"Before deletion: {original_count:,} rows")

# Keep only complete rows
final_merged_df_with_metadata_clean = final_merged_df_with_metadata[~incomplete_rows_mask].copy()
final_merged_df_clean = final_merged_df_with_metadata_clean[final_column_order].copy()

new_count = len(final_merged_df_with_metadata_clean)
deleted_count = original_count - new_count

print(f"After deletion:  {new_count:,} rows")
print(f"Deleted:         {deleted_count:,} rows ({(deleted_count/original_count*100):.1f}%)")

# --- Step 3: Final Verification ---
print(f"\n" + "=" * 60)
print("FINAL VERIFICATION")
print("=" * 60)

print("Checking for any remaining empty cells...")
verification_results = analyze_empty_cells(final_merged_df_with_metadata_clean, data_columns)

total_remaining_empty = sum(verification_results.values())
if total_remaining_empty == 0:
    print(f"\n SUCCESS: All {len(data_columns)} data columns are now complete!")
    print(f" Final clean dataset: {len(final_merged_df_with_metadata_clean):,} rows")
else:
    print(f"\n WARNING: {total_remaining_empty} empty cells still found")

# --- Step 4: Update Global Variables ---
print(f"\n" + "=" * 60)
print("UPDATING DATASETS")
print("=" * 60)

# Update the global dataframes with clean versions
final_merged_df_with_metadata = final_merged_df_with_metadata_clean
final_merged_df = final_merged_df_clean

print(" Updated final_merged_df_with_metadata (clean)")
print(" Updated final_merged_df (clean)")
print(f" Ready for final processing with {len(final_merged_df):,} complete rows")

print("=" * 80)


EMPTY CELL ANALYSIS AND CLEANUP FOR 8 DATA COLUMNS
CURRENT STATE ANALYSIS:
Total rows: 5,048

Column                              Empty    %      Filled  
------------------------------------------------------------
Goodreads Sample Quote              0        0.0  % 5,048   
Google Books Length Matched Snippet 0        0.0  % 5,048   
T50 Quote                           235      4.7  % 4,813   
Non-Literary Baseline               25       0.5  % 5,023   
Goodreads Popular Quote             0        0.0  % 5,048   
Google Books Page Text              0        0.0  % 5,048   
T50 Full Context                    235      4.7  % 4,813   
T50 Quote-Free Context              235      4.7  % 4,813   
T50 Quote-Free Context Length Matched 237      4.7  % 4,811   

SUMMARY                             Count    %
--------------------------------------------------
Rows with missing data              237      4.7%
Complete rows                       4,811    95.3%

REMOVING INCOMPLETE ROWS
Before 

In [69]:
# --- 5. Pre-sampling Steps ---

# --- 5.1. Remove duplicates based on "Title of Goodreads Sample Book" ---
print(f"Checking for duplicate book titles before final sampling...")
initial_count_before_title_dedup = len(final_merged_df_with_metadata)

# Check for duplicates in the title column
title_duplicates = final_merged_df_with_metadata['Title of Goodreads Sample Book'].duplicated().sum()
if title_duplicates > 0:
    print(f"Found {title_duplicates} duplicate book titles. Removing duplicates (keeping first occurrence)...")
    final_merged_df_with_metadata = final_merged_df_with_metadata.drop_duplicates(
        subset=['Title of Goodreads Sample Book'], 
        keep='first'
    ).copy()
    removed_count = initial_count_before_title_dedup - len(final_merged_df_with_metadata)
    print(f"Removed {removed_count} rows due to duplicate book titles.")
else:
    print("No duplicate book titles found.")

# --- 5.2. Fix data type inconsistencies in likes columns ---
print(f"Fixing data types for likes columns...")

# Convert both likes columns to integers, handling NaN values
final_merged_df_with_metadata['Likes of Sample Quote'] = pd.to_numeric(
    final_merged_df_with_metadata['Likes of Sample Quote'], 
    errors='coerce'
).fillna(0).astype(int)

final_merged_df_with_metadata['Likes of Popular Quote'] = pd.to_numeric(
    final_merged_df_with_metadata['Likes of Popular Quote'], 
    errors='coerce'
).fillna(0).astype(int)

print("Data types fixed: both likes columns are now integers.")
print("-" * 30)

# --- 5.3. Final Sampling ---
print(f"Performing final sampling to get 5000 rows from the remaining {len(final_merged_df_with_metadata)} rows...")
if len(final_merged_df_with_metadata) >= 5000:
    # Randomly sample 5000 rows, use random_state for reproducibility
    final_merged_df_with_metadata = final_merged_df_with_metadata.sample(n=5000, random_state=42)
    print("Successfully sampled 5000 rows.")
else:
    print(f"Warning: Fewer than 5000 rows ({len(final_merged_df_with_metadata)}) remain after cleaning. Using all available rows.")

# Recreate the text-only dataframe from the sampled one to ensure they match perfectly
final_merged_df = final_merged_df_with_metadata[final_column_order].copy()
print(f"Final dataset size: {len(final_merged_df)} rows.")
print("-" * 30)


# --- 6. Save the Pre-Deduplication DataFrames ---
output_path_text_only = 'ALOFT_before_semantic_deduplication.csv'
output_path_with_metadata = 'ALOFT_with_metadata_before_semantic_deduplication.csv'

final_merged_df.to_csv(output_path_text_only, index=False)
final_merged_df_with_metadata.to_csv(output_path_with_metadata, index=False)

print(f"Successfully saved the pre-deduplication datasets:")
print(f"1. Text-only (pre-dedup) data saved to: {output_path_text_only}")
print(f"2. Data with metadata (pre-dedup) saved to: {output_path_with_metadata}")

# --- 7. Display Final Data ---
print("\nHead of the final cleaned and sampled data:")
display(final_merged_df_with_metadata.head(2))

Checking for duplicate book titles before final sampling...
Found 5 duplicate book titles. Removing duplicates (keeping first occurrence)...
Removed 5 rows due to duplicate book titles.
Fixing data types for likes columns...
Data types fixed: both likes columns are now integers.
------------------------------
Performing final sampling to get 5000 rows from the remaining 4806 rows...
Final dataset size: 4806 rows.
------------------------------
Successfully saved the pre-deduplication datasets:
1. Text-only (pre-dedup) data saved to: ALOFT_before_semantic_deduplication.csv
2. Data with metadata (pre-dedup) saved to: ALOFT_with_metadata_before_semantic_deduplication.csv

Head of the final cleaned and sampled data:


,Goodreads Sample Quote,Google Books Length Matched Snippet,T50 Quote,T50 Quote-Free Context Length Matched,Non-Literary Baseline,Goodreads Popular Quote,Google Books Page Text,T50 Full Context,T50 Quote-Free Context,Likes of Sample Quote,Likes of Popular Quote,Author of Goodreads Sample Book,Title of Goodreads Sample Book,Language of Author of Goodreads Sample Book,Publication Date of Goodreads Sample Book,Genre of Goodreads Sample Book
0,"Houses, I have come to believe, like love, like nature herself, should not reassure, should not attempt to soothe, or give comfort, but should, rather,.","Who was dead, I mean. It was October 1937, a fine, crisp day, and the air in London had a sort of smoky quality to it.","Without music, life would be a mistake.",30 Errors of haste are seldom committed singly.,The Fulton County Grand Jury said Friday an investigation of Atlanta's recent primary election produced `` no evidence '' that any irregularities took place.,"If you want to know what a man's like, take a good look at how he treats his inferiors, not his equals.","It was at a funeral I first saw her, did she ever tell you that? And do you know, I can't remember whose it was! Who was dead, I mean. It was October 1937, a fine, crisp day, and the air in London had a sort of smoky quality to it. The leaves drifting off the chestnuts along Jubilee Road heaped ...","— A dentist's question. 30 Errors of haste are seldom committed singly. The first time a man always docs too much. And precisely on that account he commits a second error, and then he does too little. 31 The trodden worm curls up. This testifies to its caution. It thus reduces its chances of bei...","— A dentist's question. 30 Errors of haste are seldom committed singly. The first time a man always docs too much. And precisely on that account he commits a second error, and then he does too little. 31 The trodden worm curls up. This testifies to its caution. It thus reduces its chances of bei...",1,95186,Patrick Mcgrath,Dr. Haggard'S Disease,United Kingdom,1993,Fiction
1,"It's not something you think about, it just happens. You fall into the orbit of friends and familiar faces. You don't even have to like each other. You just have to be there to remind each other that you survived and that this is real. I'm sure there's a scientific name for it. The old fighters ...","I kick a rock down the slope and follow it as it tumbles ahead of me into the valley. As it goes, I spot something on the trail ahead. Bend down to pick it up. Okay, I might not know where I am, but I know I'm being fucked with. What I'm holding is a dusty pack of Maledictions. But no lighter.","The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.","It always puts me in mind of the country that Emily and her father travelled through, in The Mysteries of Udolpho.","City Executive Committee, which had over-all charge of the election, `` deserves the praise and thanks of the City of Atlanta '' for the manner in which the election was conducted. The September-October term jury had been charged by Fulton Superior Court Judge Durwood Pye to investigate reports ...","Friendship … Is born at the moment when one man says to another ""What! You too? I thought that no one but myself …","SO FAR, BEING dead is about as much fun as a barbed-wire G-string. Yes, there is such a thing. They invented it in Hell, which is where I am. I already said I was dead. Where else would I be? Try to keep up. Where was I? I was talking about fun. First off, there's the fact that I'm really, no sh...","She and your brother choose to go, and you will be only getting ill will. "" Catherine submitted, and though sorry to think that Isabella should be doing wrong, felt greatly relieved by Mr. Allen's approbation of her own conduct, and truly rejoiced to be preserved by his advice from the danger of...","She and your brother choose to go, and you will be only getting ill will. "" Catherine submitted,